# Jarak Titik Genangan ke Fasilitas Kesehatan Terdekat via OSRM

Notebook ini menghitung jarak jaringan jalan aktual dari setiap titik genangan
ke fasilitas kesehatan (RS, puskesmas, klinik) terdekat menggunakan OSRM Route
Service, menggantikan jarak Haversine (garis lurus) yang sebelumnya dipakai di
`severity/index.py`.

Hasilnya disimpan sebagai kolom `dist_faskes_m` di `floods.csv`, sehingga
`severity/index.py` bisa langsung memakainya tanpa perlu menghitung ulang
saat runtime.

**Mengapa OSRM, bukan Haversine?**
Jarak garis lurus antara titik genangan dan faskes bisa sangat menyesatkan
di kota seperti Surabaya karena: sungai/rel kereta yang memotong jalur langsung,
jalan satu arah, dan bentuk jaringan jalan yang tidak radial. Jarak jaringan
jalan memberikan estimasi yang lebih realistis untuk indikator C3 (kedekatan
fasilitas kesehatan), yang intinya mengukur seberapa sulit akses ke layanan
medis saat genangan terjadi.

**Input:**
- `floods.csv` (33 titik genangan)
- `faskes.csv` (404 fasilitas kesehatan)

**Output:**
- `floods.csv` diperbarui dengan kolom `dist_faskes_m` (jarak OSRM ke faskes
  terdekat, dalam meter)
- `faskes_distance_report.csv` (detail: tiap genangan dipasangkan dengan faskes
  mana, jarak OSRM vs Haversine, untuk transparansi di Bab 5)


In [4]:
%pip install -q requests pandas numpy tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import requests
import pandas as pd
import numpy as np
import time
from pathlib import Path
from tqdm import tqdm

# ===== KONFIGURASI =====
OSRM_BASE = "https://router.project-osrm.org"
REQUEST_DELAY_S = 0.5  # jeda antar-request
MAX_RETRIES = 3
RETRY_WAIT_S = 5

# Jumlah faskes kandidat terdekat (Haversine) yang dicek via OSRM per titik genangan.
# Tidak perlu cek semua 404 faskes via OSRM -- cukup ambil K terdekat secara
# Haversine, lalu hitung jarak OSRM hanya untuk K kandidat ini.
# Faskes terdekat secara Haversine hampir selalu juga terdekat secara OSRM.
K_CANDIDATES = 5

DATA_DIR = Path("../backend/app/data")
OUTPUT_DIR = Path("output_faskes_dist") 
OUTPUT_DIR.mkdir(exist_ok=True)

EARTH_RADIUS_M = 6_371_000.0

print("Setup selesai.")

Setup selesai.


## 1. Muat Data

In [21]:
floods_df = pd.read_csv(DATA_DIR / "floods-skenario1.csv")
floods_df = floods_df.rename(columns={"Latitude": "lat", "Longitude": "lon", "Ketinggian (cm)": "ketinggian_cm"})
floods_df["lat"] = pd.to_numeric(floods_df["lat"], errors="coerce")
floods_df["lon"] = pd.to_numeric(floods_df["lon"], errors="coerce")
floods_df = floods_df.dropna(subset=["lat", "lon"]).reset_index(drop=True)
if "id" not in floods_df.columns:
    floods_df.insert(0, "id", [f"F_{i:04d}" for i in range(len(floods_df))])

faskes_df = pd.read_csv(DATA_DIR / "faskes.csv")
faskes_df["lat"] = pd.to_numeric(faskes_df["lat"], errors="coerce")
faskes_df["lon"] = pd.to_numeric(faskes_df["lon"], errors="coerce")
faskes_df = faskes_df.dropna(subset=["lat", "lon"]).reset_index(drop=True)

print(f"Titik genangan: {len(floods_df)}")
print(f"Fasilitas kesehatan: {len(faskes_df)}")

Titik genangan: 34
Fasilitas kesehatan: 404


## 2. Pre-filter Kandidat Faskes Terdekat (Haversine)

Untuk tiap titik genangan, hitung jarak Haversine ke semua 404 faskes, lalu
ambil K terdekat. Ini mengurangi jumlah query OSRM dari 33 x 404 = 13.332
menjadi 33 x 5 = 165 request saja.

In [22]:
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return EARTH_RADIUS_M * 2 * np.arcsin(np.sqrt(a))


flood_lats = floods_df["lat"].values
flood_lons = floods_df["lon"].values
faskes_lats = faskes_df["lat"].values
faskes_lons = faskes_df["lon"].values

# Matriks jarak Haversine (n_floods x n_faskes)
hav_matrix = np.zeros((len(floods_df), len(faskes_df)))
for i in range(len(floods_df)):
    hav_matrix[i] = haversine(flood_lats[i], flood_lons[i], faskes_lats, faskes_lons)

# Untuk tiap flood, ambil indeks K faskes terdekat
candidates = []
for i in range(len(floods_df)):
    sorted_idx = np.argsort(hav_matrix[i])[:K_CANDIDATES]
    candidates.append(sorted_idx.tolist())
    
print(f"Kandidat per titik genangan: {K_CANDIDATES}")
print(f"Total query OSRM yang dibutuhkan: {len(floods_df) * K_CANDIDATES}")

Kandidat per titik genangan: 5
Total query OSRM yang dibutuhkan: 170


## 3. Query OSRM Route untuk Tiap Pasangan

Pakai OSRM Route service (`/route/v1/driving/`) untuk tiap pasangan
(genangan, faskes kandidat). Ambil `distance` (meter) dari respons.

In [23]:
def osrm_distance(lat1, lon1, lat2, lon2, max_retries=MAX_RETRIES):
    """Hitung jarak jaringan jalan via OSRM antara dua titik. Return meter atau None."""
    coords = f"{lon1:.6f},{lat1:.6f};{lon2:.6f},{lat2:.6f}"
    url = f"{OSRM_BASE}/route/v1/driving/{coords}?overview=false&steps=false"
    
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                data = r.json()
                if data.get("code") == "Ok" and data.get("routes"):
                    return data["routes"][0]["distance"]  # meter
            # Rate limit atau error lain
            if r.status_code == 429:
                time.sleep(RETRY_WAIT_S * 2)
            else:
                time.sleep(RETRY_WAIT_S)
        except Exception as e:
            time.sleep(RETRY_WAIT_S)
    return None


results = []

for i in tqdm(range(len(floods_df)), desc="Querying OSRM"):
    flood = floods_df.iloc[i]
    best_osrm_dist = np.inf
    best_faskes_idx = -1
    best_faskes_name = None
    best_hav_dist = np.inf
    
    for j in candidates[i]:
        fk = faskes_df.iloc[j]
        osrm_dist = osrm_distance(flood["lat"], flood["lon"], fk["lat"], fk["lon"])
        
        if osrm_dist is not None and osrm_dist < best_osrm_dist:
            best_osrm_dist = osrm_dist
            best_faskes_idx = j
            best_faskes_name = fk.get("name", "N/A")
            best_hav_dist = hav_matrix[i, j]
        
        time.sleep(REQUEST_DELAY_S)
    
    # Fallback ke Haversine jika semua OSRM query gagal
    if best_faskes_idx == -1:
        nearest_hav = np.argmin(hav_matrix[i])
        best_osrm_dist = hav_matrix[i, nearest_hav]
        best_faskes_idx = nearest_hav
        best_faskes_name = faskes_df.iloc[nearest_hav].get("name", "N/A")
        best_hav_dist = best_osrm_dist
        fallback = True
    else:
        fallback = False
    
    results.append({
        "flood_id": flood["id"],
        "nearest_faskes_name": best_faskes_name,
        "nearest_faskes_idx": best_faskes_idx,
        "dist_faskes_osrm_m": round(best_osrm_dist, 2),
        "dist_faskes_haversine_m": round(best_hav_dist, 2),
        "ratio_osrm_hav": round(best_osrm_dist / max(best_hav_dist, 1), 3),
        "fallback_haversine": fallback,
    })

results_df = pd.DataFrame(results)
n_fallback = results_df["fallback_haversine"].sum()
print(f"\nSelesai. Fallback Haversine: {n_fallback} dari {len(results_df)}")
print(f"Rata-rata rasio OSRM/Haversine: {results_df['ratio_osrm_hav'].mean():.3f}")
results_df.head(10)

Querying OSRM: 100%|██████████| 34/34 [04:10<00:00,  7.37s/it]


Selesai. Fallback Haversine: 0 dari 34
Rata-rata rasio OSRM/Haversine: 2.337


,flood_id,nearest_faskes_name,nearest_faskes_idx,dist_faskes_osrm_m,dist_faskes_haversine_m,ratio_osrm_hav,fallback_haversine
0,F_0000,Citra Medical Center,246,171.3,134.80,1.271,False
1,F_0001,Posyandu Kenanga,37,465.0,178.94,2.599,False
2,F_0002,Puskesmas Ngagel Rejo,175,1093.5,546.77,2.000,False
3,F_0003,Klinik Pratama Sebelas Medika,59,518.2,167.71,3.090,False
4,F_0004,KLINIK SANTA MARIA TAK BERCELA,102,105.4,73.46,1.435,False
5,F_0005,Puskesmas Pembantu Baratajaya,158,146.9,150.42,0.977,False
6,F_0006,Puskesmas Simomulyo,316,421.4,365.11,1.154,False
7,F_0007,Puskesmas Pembantu Simohilir,305,257.6,151.76,1.697,False
8,F_0008,Puskesmas Sidosermo,296,1926.8,837.41,2.301,False
9,F_0009,RSIA Kendangsari,309,0.0,0.01,0.000,False


## 4. Update floods.csv dan Simpan

In [24]:
# Merge jarak OSRM ke floods_df
floods_updated = floods_df.copy()
dist_map = dict(zip(results_df["flood_id"], results_df["dist_faskes_osrm_m"]))
floods_updated["dist_faskes_m"] = floods_updated["id"].map(dist_map)

# Simpan floods.csv yang sudah diupdate
floods_out = floods_updated.copy()
# Kembalikan nama kolom asli untuk konsistensi dengan format CSV backend
floods_out = floods_out.rename(columns={
    "lat": "Latitude", "lon": "Longitude", "ketinggian_cm": "Ketinggian (cm)"
})
cols_to_save = [c for c in floods_out.columns if c != "id"]  # id di-generate oleh backend
floods_out[cols_to_save].to_csv(OUTPUT_DIR / "floods_with_faskes_dist.csv", index=False)

# Simpan laporan detail untuk transparansi Bab 5
results_df.to_csv(OUTPUT_DIR / "faskes_distance_report.csv", index=False)

print(f"Tersimpan: {OUTPUT_DIR}/floods_with_faskes_dist.csv")
print(f"Tersimpan: {OUTPUT_DIR}/faskes_distance_report.csv")
print()
print("Langkah berikutnya:")
print("1. Copy floods_with_faskes_dist.csv ke backend/app/data/floods.csv")
print("   (atau merge kolom dist_faskes_m ke floods.csv yang sudah ada)")
print("2. Update severity/index.py: pakai kolom dist_faskes_m dari floods")
print("   alih-alih menghitung Haversine di runtime")

Tersimpan: output_faskes_dist/floods_with_faskes_dist.csv
Tersimpan: output_faskes_dist/faskes_distance_report.csv

Langkah berikutnya:
1. Copy floods_with_faskes_dist.csv ke backend/app/data/floods.csv
   (atau merge kolom dist_faskes_m ke floods.csv yang sudah ada)
2. Update severity/index.py: pakai kolom dist_faskes_m dari floods
   alih-alih menghitung Haversine di runtime


## 5. Statistik Perbandingan OSRM vs Haversine

Tabel ini berguna untuk Bab 5 (transparansi implementasi): menunjukkan
seberapa besar perbedaan jarak jaringan jalan vs garis lurus.

In [25]:
non_fallback = results_df[~results_df["fallback_haversine"]]
if len(non_fallback) > 0:
    print("Statistik rasio OSRM / Haversine (non-fallback):")
    print(f"  Min  : {non_fallback['ratio_osrm_hav'].min():.3f}")
    print(f"  Mean : {non_fallback['ratio_osrm_hav'].mean():.3f}")
    print(f"  Max  : {non_fallback['ratio_osrm_hav'].max():.3f}")
    print(f"  Std  : {non_fallback['ratio_osrm_hav'].std():.3f}")
    print()
    print("Interpretasi:")
    print("  Rasio > 1 berarti jarak jaringan jalan lebih jauh dari garis lurus")
    print("  (yang selalu terjadi kecuali jalan lurus sempurna).")
    print("  Rasio 1.3-1.5 tipikal untuk kota grid, >1.5 menandakan hambatan")
    print("  signifikan (sungai, rel, jalan satu arah).")
else:
    print("Semua query fallback ke Haversine, tidak ada statistik perbandingan.")

Statistik rasio OSRM / Haversine (non-fallback):
  Min  : 0.000
  Mean : 2.337
  Max  : 15.460
  Std  : 2.673

Interpretasi:
  Rasio > 1 berarti jarak jaringan jalan lebih jauh dari garis lurus
  (yang selalu terjadi kecuali jalan lurus sempurna).
  Rasio 1.3-1.5 tipikal untuk kota grid, >1.5 menandakan hambatan
  signifikan (sungai, rel, jalan satu arah).


## Catatan Penting

- Setelah kolom `dist_faskes_m` tersedia di `floods.csv`, fungsi
  `_distance_to_nearest_faskes()` di `severity/index.py` bisa diganti:
  alih-alih menghitung Haversine saat runtime, langsung baca kolom ini.
  Ini membuat runtime SI lebih cepat dan hasilnya konsisten dengan
  jarak jaringan jalan.
- Rasio OSRM/Haversine di `faskes_distance_report.csv` dilaporkan di
  Bab 5 untuk menunjukkan bahwa penggunaan OSRM memberikan estimasi
  yang lebih realistis.
- Sama seperti Notebook 4 (matriks jarak), notebook ini dijalankan
  sekali secara offline saat dataset berubah, bukan setiap runtime.